# #1

In [5]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# 1. 데이터셋 로드
df = pd.read_csv('tmdb_5000_movies.csv')

# 데이터 구조 및 컬럼 확인
print("데이터셋 크기(Shape):", df.shape)
print("전체 컬럼 목록:\n", df.columns.tolist())

# 주요 컬럼 설명:
# - id: 영화의 고유 ID
# - title: 영화 제목
# - genres: 영화가 속한 장르 (JSON 형태의 문자열)
# - keywords: 영화의 키워드 (JSON 형태의 문자열)
# - vote_average: 영화의 평균 평점
# - vote_count: 평점 투표 수

데이터셋 크기(Shape): (4803, 20)
전체 컬럼 목록:
 ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count']


In [6]:
# 2. 문자열 형태의 데이터를 리스트 객체로 변환 
df['genres'] = df['genres'].apply(eval)
df['keywords'] = df['keywords'].apply(eval)

In [7]:
# 3. genres 칼럼에서 장르 이름만 추출하여 공백으로 연결된 문자열 생성
df['genres_literal'] = df['genres'].apply(lambda x: ' '.join([d['name'] for d in x]))

print(df[['title', 'genres_literal']].head())

                                      title  \
0                                    Avatar   
1  Pirates of the Caribbean: At World's End   
2                                   Spectre   
3                     The Dark Knight Rises   
4                               John Carter   

                             genres_literal  
0  Action Adventure Fantasy Science Fiction  
1                  Adventure Fantasy Action  
2                    Action Adventure Crime  
3               Action Crime Drama Thriller  
4          Action Adventure Science Fiction  


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 4. CountVectorizer를 활용한 벡터화
count_vect = CountVectorizer(min_df=1, ngram_range=(1, 1))
genre_mat = count_vect.fit_transform(df['genres_literal'])

print("벡터의 Shape:", genre_mat.shape) 
# 비교 설명: 전체 데이터(영화) 개수는 4803개이며, genres_literal을 공백으로 분리하여 
# 카운트한 고유 장르(단어)의 개수가 총 22개이므로 (4803, 22) 형태의 행렬이 완성됩니다.

# 5. 코사인 유사도 계산
genre_sim = cosine_similarity(genre_mat, genre_mat)
print("유사도 행렬의 Shape:", genre_sim.shape) 

genre_sim_sorted_ind = genre_sim.argsort()[:, ::-1]

벡터의 Shape: (4803, 22)
유사도 행렬의 Shape: (4803, 4803)


In [9]:
# 6. 특정 영화와 유사한 영화를 추천하는 함수
def get_recommendations(df, sorted_ind, title_name, top_n=10):
    title_movie = df[df['title'] == title_name]
    
    if len(title_movie) == 0:
        return "해당 영화가 데이터셋에 존재하지 않습니다."
        
    title_index = title_movie.index.values
    
    similar_indexes = sorted_ind[title_index, 1:(top_n+1)]
    
    similar_indexes = similar_indexes.reshape(-1)
    
    return df.iloc[similar_indexes][['title', 'genres_literal', 'vote_average']]


recommended_movies = get_recommendations(df, genre_sim_sorted_ind, 'The Godfather', top_n=5)
print("\n['The Godfather'와 유사한 영화 TOP 5]")
print(recommended_movies)


['The Godfather'와 유사한 영화 TOP 5]
                                               title genres_literal  \
2072                                 Freedom Writers    Crime Drama   
4063                                           Bully    Crime Drama   
3348                                          Capote    Crime Drama   
4041                                 This Is England    Drama Crime   
1946  The Bad Lieutenant: Port of Call - New Orleans    Drama Crime   

      vote_average  
2072           7.5  
4063           6.7  
3348           6.8  
4041           7.4  
1946           6.0  


# #2

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# 1. 데이터 로드
movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

# 2. 데이터 전처리
movie_ratings = pd.merge(ratings, movies, on='movieId')

ratings_matrix = movie_ratings.pivot_table(index='title', columns='userId', values='rating')

ratings_matrix.fillna(0, inplace=True)

print("피벗 테이블 생성 완료. Shape:", ratings_matrix.shape)
display(ratings_matrix.head(2)) 

피벗 테이블 생성 완료. Shape: (9719, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
'Hellboy': The Seeds of Creation (2004),0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 3. 영화 간 코사인 유사도 계산
item_sim = cosine_similarity(ratings_matrix, ratings_matrix)

item_sim_df = pd.DataFrame(data=item_sim, index=ratings_matrix.index, columns=ratings_matrix.index)

# 4. 특정 영화와 유사한 영화를 추천하는 함수
def recommend_movie(title_name, top_n=10):
    if title_name not in item_sim_df.columns:
        return "해당 영화가 존재하지 않습니다."
        
    return item_sim_df[title_name].sort_values(ascending=False)[1:top_n+1]

print(recommend_movie('Matrix, The (1999)', top_n=5))

title
Fight Club (1999)                                        0.713937
Star Wars: Episode V - The Empire Strikes Back (1980)    0.700935
Saving Private Ryan (1998)                               0.679615
Star Wars: Episode IV - A New Hope (1977)                0.663447
Star Wars: Episode VI - Return of the Jedi (1983)        0.660984
Name: Matrix, The (1999), dtype: float64


In [ ]:
# 5. 유저 기반 추천 시스템 구현
user_ratings_matrix = ratings_matrix.T

# 유저 간 코사인 유사도 계산
user_sim = cosine_similarity(user_ratings_matrix, user_ratings_matrix)
user_sim_df = pd.DataFrame(data=user_sim, index=user_ratings_matrix.index, columns=user_ratings_matrix.index)

def recommend_for_user(user_id, top_n=10):
    if user_id not in user_sim_df.index:
        return "해당 유저가 존재하지 않습니다."
    
    # 5-1. 대상 유저가 이미 시청한 영화 목록 추출
    user_seen_movies = user_ratings_matrix.loc[user_id]
    seen_movies_list = user_seen_movies[user_seen_movies > 0].index.tolist()
    
    # 5-2. 대상 유저와 가장 유사도가 높은 유저 찾기 
    most_similar_user = user_sim_df[user_id].sort_values(ascending=False).index[1]
    
    # 5-3. 가장 유사한 유저의 모든 영화 평점 추출
    sim_user_movies = user_ratings_matrix.loc[most_similar_user]
    
    # 5-4. 그 중에서 대상 유저가 아직 안 본 영화만 남기기
    sim_user_unseen = sim_user_movies.drop(seen_movies_list, errors='ignore')
    
    # 평점이 높은 순으로 정렬하여 top_n개 반환
    return sim_user_unseen.sort_values(ascending=False).head(top_n)

print("\n[유저 ID 1을 위한 추천 영화]")
print(recommend_for_user(user_id=1, top_n=5))


[유저 ID 1을 위한 추천 영화]
title
Dirty Work (1998)              5.0
Three Kings (1999)             5.0
Commitments, The (1991)        5.0
Summer of Sam (1999)           5.0
Fish Called Wanda, A (1988)    5.0
Name: 266, dtype: float64
